# **HW4 – Quantum Noise, Transpilation, Phase Estimation, and Factoring**
_Time required: ~2–3 hours (for students with Qiskit noise and algorithm experience)_

**What you’ll practice**
- Modeling quantum noise (decoherence, gate errors)
- Applying error mitigation techniques (ZNE, readout correction)
- Using Qiskit transpiler for circuit optimization
- Implementing Quantum Fourier Transform (QFT)
- Building Quantum Phase Estimation (QPE) circuits
- Simulating Shor’s algorithm for period finding and factoring
- Analyzing noise impact on deep circuits
- Connections to AI/DS (e.g., eigenvalue estimation)

**What to turn in**
- This single notebook (`HW4_YourName.ipynb`) with **all cells run**, code and short written answers filled in where prompted.

**Rules & hints**
- Use **Qiskit** (version ~1.0 or later).
- Use AerSimulator with noise models for reproducibility.
- If stuck, explain reasoning; partial credit for clear work.
- For circuits, use `qc.draw('mpl')`; for states, use `Statevector` or `DensityMatrix`.
- Use small n for QPE/Shor to avoid long sim times.


In [ ]:
# --- Setup (run me first) ---
# Install Qiskit if needed (uncomment in Colab)
# !pip install qiskit qiskit-aer qiskit-ibm-runtime matplotlib qiskit-ignis

# Import necessary modules
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, pauli_error, thermal_relaxation_error
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector, DensityMatrix, Kraus
from qiskit.providers.fake_provider import FakeJakarta
from qiskit_ibm_runtime import QiskitRuntimeService, Estimator
import numpy as np
import matplotlib.pyplot as plt

# Simulator backend
sim = AerSimulator()

# Fake backend for noise
fake_backend = FakeJakarta()

# Function to run circuit and get counts
def get_counts(circ, shots=2000, noise_model=None):
    sim_noise = AerSimulator(noise_model=noise_model)
    tc = transpile(circ, sim_noise)
    result = sim_noise.run(tc, shots=shots).result()
    return result.get_counts()

def assert_close(A, B, tol=1e-8):
    if not np.allclose(A, B, atol=tol):
        raise AssertionError(f"Not close:\n{A}\nvs\n{B}")


## Part A — Quantum Noise Models (≈25 min)

**A1.** Build a depolarizing noise model for p=0.01 on 1-qubit gates. Apply to a simple H circuit; compute noisy density matrix.  
**A2.** Add thermal relaxation (T1=50us, T2=30us) to a noise model. Simulate a delayed Z gate; check dephasing.  
**A3.** Short answer: Explain Kraus operators for Pauli noise; why CPTP.


In [ ]:
# A1. Depolarizing noise
noise_model_dep = NoiseModel()
# YOUR CODE HERE: add depolarizing_error(0.01, 1) to 'h'
qc_h = QuantumCircuit(1)
qc_h.h(0)
rho_noisy = DensityMatrix.from_instruction(qc_h).evolve(Kraus(noise_model_dep.get_noise('h', [0])))
print("Noisy density:\n", rho_noisy)

# A2. Thermal relaxation
noise_model_therm = NoiseModel()
# YOUR CODE HERE: add thermal_relaxation_error(50e-6, 30e-6, 1e-6) to 'delay'
qc_delay = QuantumCircuit(1)
qc_delay.delay(10, unit='us')  # 10us delay
qc_delay.z(0)
counts_therm = get_counts(qc_delay, noise_model=noise_model_therm)
plot_histogram(counts_therm)
plt.show()

# A3. Written answer: (K0=sqrt(1-p)I, Kx=sqrt(px)X, etc.; sum K†K=I preserves trace, positivity)


## Part B — Error Mitigation Techniques (≈30 min)

**B1.** Implement readout error mitigation on a 2-qubit Bell circuit. Build calibration matrix.  
**B2.** Apply ZNE to a noisy RX(π/2) gate; extrapolate to zero noise.  
**B3.** Short answer: How does dynamical decoupling suppress dephasing? Pros/cons of PEC.


In [ ]:
# B1. Readout mitigation on Bell
qc_bell = QuantumCircuit(2, 2)
# YOUR CODE HERE: h(0), cx(0,1), measure_all
counts_noisy = get_counts(qc_bell)  # Assume some noise
# Calibration: build matrix A where A_ij = P(meas i | prep j)
# YOUR CODE HERE: circuits for |00>, |01>, |10>, |11>; run and form A
A_inv = np.linalg.inv(A)
counts_mit = A_inv @ np.array([counts_noisy.get('00',0), ...])  # Vectorize counts
print("Mitigated counts: ", counts_mit)

# B2. ZNE on RX
from qiskit_ibm_runtime.fake_provider import FakeJakarta
backend = FakeJakarta()
qc_rx = QuantumCircuit(1)
qc_rx.rx(np.pi/2, 0)
# YOUR CODE HERE: use Estimator with resilience_level=1 for ZNE
estimator = Estimator(backend=backend, options={'resilience_level':1})
job = estimator.run(qc_rx, observables=['Z'])
result_zne = job.result()
print("ZNE <Z>: ", result_zne.values[0])

# B3. Written answer: (Idle sequences like XX cancel phase errors; PEC: probabilistic, increases variance/shots but corrects coherent errors)


## Part C — Transpilation in Qiskit (≈25 min)

**C1.** Transpile a 3-qubit GHZ circuit for FakeJakarta at level 0 and 3; compare depth.  
**C2.** Use noise-adaptive layout; simulate noisy counts at level 3.  
**C3.** Short answer: Explain transpiler passes (e.g., Unroller, Optimize1qGates); trade-offs in levels.


In [ ]:
# C1. Transpile GHZ
qc_ghz = QuantumCircuit(3)
# YOUR CODE HERE: h(0), cx(0,1), cx(1,2)
tc0 = transpile(qc_ghz, fake_backend, optimization_level=0)
tc3 = transpile(qc_ghz, fake_backend, optimization_level=3)
print("Level 0 depth: ", tc0.depth(), " Level 3: ", tc3.depth())

# C2. Noise-adaptive
tc_adapt = transpile(qc_ghz, fake_backend, optimization_level=3, layout_method='noise_adaptive')
qc_ghz_meas = tc_adapt.copy()
qc_ghz_meas.measure_all()
counts_adapt = get_counts(qc_ghz_meas, noise_model=fake_backend.noise_model)
plot_histogram(counts_adapt)
plt.show()

# C3. Written answer: (Unroller: basis gates; Optimize1q: merge rotations; levels: higher more aggressive but slower compile)


## Part D — Quantum Fourier Transform & Phase Estimation (≈30 min)

**D1.** Implement QFT for n=3; draw circuit. Verify on |001> statevector.  
**D2.** Build QPE for U=Z (phase 0.5 on |1>), t=3 precision qubits. Run and check measurement ~ '100' (0.5=4/8).  
**D3.** Short answer: Derive QFT action on |x>; why exponential precision in QPE.


In [ ]:
# D1. QFT n=3
def qft(n):
    qc = QuantumCircuit(n)
    # YOUR CODE HERE: for i in range(n): h(i); then controlled Rk
    # Add swaps
    return qc
qc_qft = qft(3)
qc_qft.draw('mpl')
plt.show()
qc_test = QuantumCircuit(3)
qc_test.x(0)  # |001>
qc_test = qc_test.compose(qc_qft)
sv_qft = Statevector(qc_test)
print("QFT|001>: ", sv_qft)

# D2. QPE for Z
t = 3  # precision
qc_qpe = QuantumCircuit(t+1, t)
# YOUR CODE HERE: h(range(t)), x(t) for |1> eigen, controlled-U^k where U=Z
# Inverse QFT on first t, measure
counts_qpe = get_counts(qc_qpe)
plot_histogram(counts_qpe)
plt.show()

# D3. Written answer: (Sums exp(2πixy/N)|y>; controlled powers give binary fraction of phase)


## Part E — Shor’s Algorithm & Applications (≈30 min)

**E1.** Implement modular exponentiation for a=2, N=15, x=1 to 7 (small circuit).  
**E2.** Use QPE for period finding on a=7 mod 15; find r=4.  
**E3.** Short answer: Factor 15 from r; QPE in quantum PCA; noise challenges for Shor.


In [ ]:
# E1. Mod exp (simplified)
def mod_exp(a, N, bits):
    qc = QuantumCircuit(bits*2)  # x and result regs
    # YOUR CODE HERE: controlled multiplies mod N
    return qc

# E2. Shor period
a=7; N=15
# Build QPE with U = mod_mult by a
counts_shor = get_counts(qc_shor)
phase = int(max(counts_shor, key=counts_shor.get),2) / 8  # For t=3
from fractions import Fraction
r = Fraction(phase).denominator
print("Period r: ", r)
assert r == 4

# E3. Written answer: (gcd(a^{r/2}±1, N)=3,5; QPE on Hamiltonian for eigenvalues; deep circuits accumulate errors)
